In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from lesson_1.ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
from elasticsearch import Elasticsearch

In [3]:
es = Elasticsearch("http://localhost:9200")

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
documents = load_faq_data()
documents[:2]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [6]:
texts = [doc["question"].removeprefix('Course: ') + " " + doc["answer"] for doc in documents]
texts[:3]

["When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 "What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up front. The cour

In [7]:
#chunk the dataset (texts) by batch. There are 27 batches given texts size is 13++ and each batch is 50. Encode by batch. 
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors), len(vectors[0])

  0%|          | 0/27 [00:00<?, ?it/s]

(1349, 384)

In [8]:
X = np.array(vectors)
X.shape #1346 vectors (rows) with 384 dimensions (columns)

# Each vector is the semantic "fingerprint" of its corresponding text chunk. So:
    # vectors[0] captures the meaning of the course start date chunk
    # vectors[1] captures the meaning of the prerequisites chunk

(1349, 384)

In [9]:
dimension = len(X[0])

def create_vector_index(documents, X):

    # Delete first in case the index existed:
    es.indices.delete(index="embedded_faq_zoomcamp", ignore_unavailable=True)

    # Create index:
    index_settings = {
        "mappings": {
            "properties": {
                "course":   {"type": "keyword"}
                ,"Q": {"type": "text"}
                ,"A":   {"type": "text"}
                ,"vector": {
                    "type":       "dense_vector"
                    ,"dims":       len(X[0])
                    ,"index":      True
                    ,"similarity": "cosine"
                }
            }
        }
    }

    es.indices.create(index="embedded_faq_zoomcamp", body=index_settings)

    # Index documents with vectors:
    for i, doc in enumerate(documents):
        es.index(
            index="embedded_faq_zoomcamp",
            id=doc["id"],
            body={
                "course": doc["course"]
                ,"Q": doc["question"].removeprefix('Course: ')
                ,"A": doc["answer"]
                ,"vector": X[i].tolist()
            }
        )
    
    # Force refresh so all docs are committed before counting
    es.indices.refresh(index="embedded_faq_zoomcamp")

    doc_count = es.count(index="embedded_faq_zoomcamp")['count']
    
    print(f'Vector index created and indexed {doc_count} docs')

In [10]:
create_vector_index(documents, X)

Vector index created and indexed 1349 docs


In [11]:
filter_dict = {"course": "llm-zoomcamp"}

response = es.search(
    index="embedded_faq_zoomcamp"
    ,body={
        "query": {
            "bool": {
                "must": {"match_all": {}}
                ,"filter": {
                    "term": filter_dict
                }
            }
        }
        ,"size": 2
    }
)

for hit in response["hits"]["hits"]:
    print(hit["_id"],hit["_score"])
    print(hit["_source"])
    print("---")

74eb249bbf 1.0
{'course': 'llm-zoomcamp', 'Q': 'I just discovered the course. Can I still join?', 'A': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'vector': [-0.059979673475027084, -0.05774865299463272, -0.03097953461110592, 0.019150355830788612, 0.024852700531482697, -0.019456513226032257, -0.08058656752109528, -0.061115678399801254, -0.06444694101810455, -0.042277153581380844, -0.04218591749668121, -0.012241659685969353, 0.042575858533382416, 0.03009413555264473, 0.015766294673085213, -0.03331268951296806, -0.0864262506365776, -0.02962839975953102, 0.05685662478208542, 0.0066210185177624226, -0.057525768876075745, 0.03197609260678291, -0.017901422455906868, 0.009050605818629265, -0.014255526475608349, -0.021934421733021736, 0.03953806683421135, 0.08630133420228958, 0.006488617975264788, -0.035758163779973984, -0.03371448814868927, 0.04240158572793007, 0.028510570526123047, 0.00923612155020237, 0.0255021639

In [12]:
response = es.search(
    index="embedded_faq_zoomcamp"
    ,body={
        "query": {"match_all": {}}
        ,"size": 2
    }
)

for hit in response["hits"]["hits"]:
    print(hit["_source"])
    print("---")

{'course': 'data-engineering-zoomcamp', 'Q': 'GCP BQ: When querying two different tables, external and materialized, why do you get the same result with count(distinct(*))?', 'A': 'You need to uncheck cache preferences in query settings\n\n<{IMAGE:image_1}>\n\n<{IMAGE:image_2}>', 'vector': [-0.009066523984074593, -0.031092854216694832, 0.0449543260037899, 0.09339790046215057, -0.09583336114883423, -0.015956932678818703, 0.020994465798139572, 0.02904270403087139, 0.10053453594446182, 0.011164797469973564, 0.042819131165742874, -0.02495666779577732, 0.11868233233690262, -0.04045592620968819, 0.010904612950980663, 0.045152682811021805, -0.020939815789461136, -0.07664709538221359, -0.028376908972859383, -0.0450739860534668, -0.02882247231900692, -0.024442780762910843, 0.04829666391015053, -0.04198399558663368, 0.04039612412452698, -0.019824935123324394, -0.09040568023920059, -0.023480508476495743, 0.08709415793418884, -0.07224556058645248, -0.03740613907575607, 0.010663016699254513, -0.065